# Prueba corta: VS Code + Colab + Drive

**Objetivo:** comprobar que VS Code ejecuta Python en Colab, reconoce la GPU,
hace un cálculo pequeño y puede leer las carpetas del proyecto en Drive.
Duración: unos segundos de cálculo, más el inicio de sesión y montaje de Drive.
No instala paquetes, descarga modelos ni inicia entrenamientos o runs de MLflow.

**Antes de ejecutar:** abre este archivo en VS Code y selecciona arriba a la derecha
**Select Kernel → Colab → New Colab Server**. Inicia sesión con la misma cuenta
de Google que usas para `MAIA_TESIS` y elige una máquina con GPU disponible.
Para esta prueba basta cualquier GPU CUDA; A100 es la usada en el entrenamiento.
`Open Colab Web` abre la interfaz web; aquí vamos a ejecutar desde VS Code.

Ejecuta las cinco celdas de código en orden. Si alguna falla, detente en esa celda.
No uses el intérprete Python local como kernel para esta prueba.

## 1. Identificar el entorno

Debe mostrar Linux y la versión de PyTorch del runtime de Colab.


In [1]:
import json
import platform
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    import google.colab
    import torch
except ImportError as error:
    raise RuntimeError("Selecciona el kernel de Colab en VS Code antes de ejecutar.") from error

if not Path("/content").is_dir():
    raise RuntimeError("No se encuentra /content. Revisa que el kernel sea el de Colab.")

MAIA_PROBE_SESSION_ID = uuid.uuid4().hex[:12]
MAIA_PROBE = {
    "session_id": MAIA_PROBE_SESSION_ID,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "system": platform.system(),
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "gpu_ok": False,
    "drive_ok": False,
}
print(json.dumps(MAIA_PROBE, indent=2, ensure_ascii=False))
print("ENTORNO_COLAB_OK")


{
  "session_id": "8b85e846b1db",
  "started_at_utc": "2026-11-04T15:47:44.537748+00:00",
  "system": "Linux",
  "python": "3.13.16",
  "python_executable": "/usr/bin/python3",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "gpu_ok": false,
  "drive_ok": false
}
ENTORNO_COLAB_OK


## 2. Comprobar la GPU y un cálculo pequeño

Se multiplica una matriz de 16 × 16 en la GPU y se compara con el cálculo en CPU.
Debe aparecer `CALCULO_GPU_OK` junto al nombre de la GPU. No se mide rendimiento.
Si no hay GPU, selecciona otro servidor de Colab con GPU y vuelve a ejecutar desde 1.


In [2]:
if not torch.cuda.is_available():
    raise RuntimeError("Este servidor no tiene GPU CUDA. Selecciona New Colab Server con GPU.")

gpu = torch.cuda.get_device_properties(0)
matrix_cpu = torch.arange(256, dtype=torch.float32).reshape(16, 16)
matrix_gpu = matrix_cpu.to("cuda:0")
expected = matrix_cpu @ matrix_cpu.T
actual = (matrix_gpu @ matrix_gpu.T).cpu()
torch.cuda.synchronize()
if not torch.allclose(actual, expected, atol=1e-5, rtol=1e-5):
    raise RuntimeError("El resultado de GPU no coincide con la referencia CPU.")

MAIA_PROBE.update({
    "gpu": gpu.name,
    "gpu_memory_gib": round(gpu.total_memory / 2**30, 2),
    "is_a100": "A100" in gpu.name,
    "gpu_ok": True,
    "calculation_checksum": float(actual.sum()),
})
print("GPU:", MAIA_PROBE["gpu"])
print("Memoria total (GiB):", MAIA_PROBE["gpu_memory_gib"])
print("Es A100:", MAIA_PROBE["is_a100"])
print("CALCULO_GPU_OK")


GPU: Tesla T4
Memoria total (GiB): 14.56
Es A100: False
CALCULO_GPU_OK


## 3. Montar Drive y comprobar las rutas del proyecto

Autoriza el acceso cuando Colab lo solicite. Se comprueban las carpetas y, si existe,
se lee el marcador pequeño del modelo base. Esta celda no escribe archivos en Drive.

Si no aparece el diálogo, revisa las notificaciones de VS Code y el navegador.
La extensión también ofrece el comando `Colab: Mount Google Drive to Server...`.


In [3]:
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

my_drive = Path("/content/drive/MyDrive")
required_paths = {
    "resultados": my_drive / "MAIA_TESIS",
    "datos_compartidos": my_drive / "MAIA_PROYECTO",
    "ids_grupo15": my_drive / "MAIA_TESIS/TrainingData_Grupo15",
}
checks = {name: path.is_dir() for name, path in required_paths.items()}
for name, path in required_paths.items():
    print("OK" if checks[name] else "FALTA", name, "→", path)
MAIA_PROBE["drive_checks"] = checks
if not all(checks.values()):
    raise FileNotFoundError("Revisa la cuenta de Google y las carpetas marcadas como FALTA.")

marker = my_drive / "MAIA_TESIS/a4/g15_a4_v0_seed42_mlflow_01/base/TRAINING_COMPLETE.json"
MAIA_PROBE["base_marker_readable"] = False
if marker.is_file():
    with marker.open(encoding="utf-8") as stream:
        base_marker = json.load(stream)
    MAIA_PROBE["base_marker_readable"] = True
    print("Marcador del modelo base leído correctamente.")
else:
    print("Carpetas accesibles; el marcador del modelo base no está en la ruta esperada.")

MAIA_PROBE["drive_ok"] = True
print("DRIVE_OK")


Mounted at /content/drive
OK resultados → /content/drive/MyDrive/MAIA_TESIS
OK datos_compartidos → /content/drive/MyDrive/MAIA_PROYECTO
OK ids_grupo15 → /content/drive/MyDrive/MAIA_TESIS/TrainingData_Grupo15
Marcador del modelo base leído correctamente.
DRIVE_OK


## 4. Resumen de la prueba

El resultado esperado es `PRUEBA_COLAB_VSCODE_OK`. Este resultado valida GPU y Drive.
Revisa también `base_marker_readable`: `true` confirma lectura del archivo del modelo
base existente. Una GPU diferente de A100 sirve para esta comprobación inicial.


In [4]:
if not MAIA_PROBE["gpu_ok"] or not MAIA_PROBE["drive_ok"]:
    raise RuntimeError("Falta completar la comprobación de GPU o Drive.")
MAIA_PROBE["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
print(json.dumps(MAIA_PROBE, indent=2, ensure_ascii=False))
print("PRUEBA_COLAB_VSCODE_OK")


{
  "session_id": "8b85e846b1db",
  "started_at_utc": "2026-11-04T15:47:44.537748+00:00",
  "system": "Linux",
  "python": "3.13.16",
  "python_executable": "/usr/bin/python3",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "gpu_ok": true,
  "drive_ok": true,
  "gpu": "Tesla T4",
  "gpu_memory_gib": 14.56,
  "is_a100": false,
  "calculation_checksum": 66672640.0,
  "drive_checks": {
    "resultados": true,
    "datos_compartidos": true,
    "ids_grupo15": true
  },
  "base_marker_readable": true,
  "completed_at_utc": "2026-11-04T15:48:42.371317+00:00"
}
PRUEBA_COLAB_VSCODE_OK


## 5. Dejar una variable para probar el acceso de Codex después

Ejecuta esta celda dos veces: debe mostrar contador 1 y después 2, con el mismo
`session_id`. Así comprobamos que las variables persisten en el kernel seleccionado.

Guarda el notebook con `Ctrl+S`. Para el siguiente paso conservaremos abierto este
notebook y comprobaremos si Codex puede leer `MAIA_KERNEL_PROBE` mediante el puente
de ejecución, antes de lanzar cualquier entrenamiento. Esta celda todavía no conecta
Codex por sí sola. Si reinicias el kernel, las variables se pierden.

Al terminar las pruebas de conexión y cuando ya no uses el servidor, puedes liberar
el runtime con `Colab: Remove Server`.

Referencia: [guía oficial de Colab para VS Code](https://github.com/googlecolab/colab-vscode/wiki/User-Guide).


In [7]:
MAIA_KERNEL_COUNTER = globals().get("MAIA_KERNEL_COUNTER", 0) + 1
MAIA_KERNEL_PROBE = {
    "session_id": MAIA_PROBE_SESSION_ID,
    "counter": MAIA_KERNEL_COUNTER,
    "gpu": MAIA_PROBE["gpu"],
    "gpu_ok": MAIA_PROBE["gpu_ok"],
    "drive_ok": MAIA_PROBE["drive_ok"],
    "status": "ready_for_codex_connection_test",
}
print(json.dumps(MAIA_KERNEL_PROBE, indent=2, ensure_ascii=False))
print("CONTADOR_KERNEL:", MAIA_KERNEL_COUNTER)


{
  "session_id": "8b85e846b1db",
  "counter": 3,
  "gpu": "Tesla T4",
  "gpu_ok": true,
  "drive_ok": true,
  "status": "ready_for_codex_connection_test"
}
CONTADOR_KERNEL: 3
